# 03 - Camera Intrinsics and Extrinsics

## Imports

In [1]:
import numpy as np

## Why Are We Talking about Cameras (Again)?

In our Computer Vision primer, we covered some of the basics of a camera and how it leverages light to create an image. This understanding covered parts of the basics of camera **intrinsics**, or the internals of the camera that tells us how it sees the world. 

For us to understand how to use Limelights (or any other cameras), particularly more than one, to perform object detection tasks, we need to not only have a strong understanding of the intrinsics, but also the camera **extrinsics**, or where the camera is located in the real world. We need to know the location of all of our cameras so we can use them together! 

It is important to note that we will be discussing a bit more math-intensive topics; do not stress if the math is a little heavier! The main point is to give you the foundational understanding, so if you get the big ideas, you are in good shape. Remember, Limelight does all of these calculations on its own; this is here so you understand a bit about how and why. 

#### Perspective Projection Recap

Recall from our discussion about perspective projection that the fundamental operation a camera does is mapping points in the real world (which is 3D) to points on a piece of paper (which is 2D), also known as perspective projection. 

From the pinhole camera geometry below:


![](../cv_primer/ref_imgs/projection_diagram.png)

We were able to recreate the perspective projection equations:

$x = f\frac{X}{Z}$, $y = f\frac{Y}{Z}$

Revisit the notebook `05-perspective-projection.ipynb` if you want to recap this information more deeply; it is worth the extra time.

What we will be walking through in this notebook is how we can relate our perspective projection to both the intrinsics and extrinsics of a camera. That is, how understanding both what's inside and outside of a camera will help us understand the position of objects in images. 

## Intrinsics

### Intrinsic Matrix

As shown by our above diagram, we map 3D points from the perspective of the camera to 2D points on an image. Our intrinsic matrix is what helps us perform this mapping, using a matrix 

$$
K = 
\begin{bmatrix}
f_x & s & c_x \\
0 & f_y & c_y \\
0 & 0 & 1
\end{bmatrix}
$$ 

Where $(c_x, c_y)$ is our **principal point** ($p$), $(f_x, f_y)$ are our **focal lengths**, where we have two different ones for $x$ and $y$ to account for some potential non-square pixels, and our **skew** $s$ which corrects for non-perpendicular axes between the camera sensor and the camera lens. We typically assume $s = 0$ (no non-perpendicular axes) and $f_x = f_y$ (no non-square pixels) for simplicity by reducing the number of variables we deal with in our matrix multiplication, leaving us with 

$$
K = 
\begin{bmatrix}
f & 0 & c_x \\
0 & f & c_y \\
0 & 0 & 1
\end{bmatrix}
$$ 

Note that typically our principal point is also $(0, 0)$ in a lot of simplified applications, but for our use cases, this simply isn't a realistic assumption given the amount of movement and different cameras we use. 

Let's look at an example matrix below: 


In [2]:
sample_k = np.array([[800, 0, 320], [0, 800, 240], [0, 0, 1]])
sample_k

array([[800,   0, 320],
       [  0, 800, 240],
       [  0,   0,   1]])

What do these numbers actually mean? A focal length of 800 pixels doesn't tell us much on its own. It is the distance from the optical center of the camera lens (the "pinhole") to the image sensor plane, measured in pixels, so it is only meaningful relative to the size of the image.

We can turn it into something more intuitive with the **field of view (FOV)**, sometimes called angular FOV or AFOV, the angle describing how much of the world the camera can see. Assuming our image resolution is $640 \times 480$, our principal point $(320, 240)$ sits exactly at the center of the image (real calibrated cameras are usually off by a few pixels, but near-center is the typical case).

**Quick derivation.** The right edge of the image is $320$ pixels from the principal point, and the principal point is $f = 800$ pixels from the camera center. That's a right triangle, and the angle at the camera center is half of the FOV, so $\tan(\text{FOV}_x / 2) = 320 / 800$. In general, with image width $W$ and height $H$:

$$
\text{FOV}_x = 2\arctan\left(\frac{W}{2f}\right) \approx 44^\circ, \qquad
\text{FOV}_y = 2\arctan\left(\frac{H}{2f}\right) \approx 33^\circ
$$

This is the same similar-triangles argument behind the projection equations; the full version, with a wide-vs-narrow focal length comparison, is in `cv_primer/05-perspective-projection.ipynb`.

**How much world is that?** An angle doesn't change with distance, but the amount of world it covers does. At depth $Z$ (along the camera's viewing direction), the camera sees a patch of the world that is

$$
\text{width}(Z) = 2Z\tan\left(\frac{\text{FOV}_x}{2}\right) = Z\cdot\frac{W}{f}, \qquad
\text{height}(Z) = Z\cdot\frac{H}{f}
$$

The two forms of the width are the same equation, because the $\tan$ and $\arctan$ cancel. At $Z = 5$ meters, our camera sees a patch $5 \cdot 640 / 800 = 4$ meters wide and $5 \cdot 480 / 800 = 3$ meters tall. This assumes a flat surface facing the camera head-on; a tilted surface covers more ground.

This is also why we call $f = 800$ "moderately zoomed in". A $44^\circ$ horizontal FOV is narrower than many webcams and wide-angle Limelight modes, which see closer to $60^\circ$ - $80^\circ$.

### Mapping from 3D Camera World to 2D Image 

We can actually use this intrinsic matrix to map 3D points in our camera coordinates to the 2D image. We use the following equation: 

$p = KP$

where 

$p = 
\begin{bmatrix}
u \\
v \\ 
1
\end{bmatrix}
$,
$P =
\begin{bmatrix}
X_c \\
Y_c \\
Z_c \\
\end{bmatrix}
$, 
$K = 
\begin{bmatrix}
f_x & s & c_x \\
0 & f_y & c_y \\
0 & 0 & 1
\end{bmatrix}
$

with $p$ as our point in the 2D plane, $K$ our intrinsic matrix, and $P$ as our point in the 3D plane (with the $c$ subscript reminding us that we are dealing with coordinates with respect to the camera). Note that $p$ is expressed in **homogeneous coordinates**, where that trailing $1$ is a placeholder for the actual scale factor, which we'll unpack in a moment.

We can write out this equation out as 

$$
\begin{bmatrix}
u \\
v \\ 
1
\end{bmatrix} = 
\begin{bmatrix}
f_x & s & c_x \\
0 & f_y & c_y \\
0 & 0 & 1
\end{bmatrix}
\begin{bmatrix}
X_c \\
Y_c \\
Z_c \\
\end{bmatrix}
$$ 

If we actually do this multiplication, we get the following representations of our 2D coordinates as 

$$
u = f_x \frac{X_c}{Z_c} + c_x
$$, 
$$
v = f_y \frac{Y_c}{Z_c} + c_y
$$

These are our perspective projection equations, taking into account the principal point! 

So, we have shown a bit more formally how the camera's intrinsic properties determine perspective projection. 





Let's make this concrete with real numbers. One subtlety worth flagging: multiplying $K$ by the raw camera coordinates $(X_c, Y_c, Z_c)$ doesn't give you $(u, v, 1)$ directly. It hands you $(u \cdot Z_c,\ v \cdot Z_c,\ Z_c)$, and you still need to divide through by that last entry (the depth) to land on actual pixel coordinates. That division is what's hiding inside the $\frac{X_c}{Z_c}$ terms above. Let's verify this with an actual point:

In [3]:
# a 3D point in camera coordinates (X_c, Y_c, Z_c)
P_c = np.array([0.5, 0.3, 5.0])

# p = K @ P_c gives us a *scaled* homogeneous point, not (u, v, 1) directly
p_scaled = sample_k @ P_c
print("K @ P_c =", p_scaled)

# dividing by the last entry (Z_c) recovers actual pixel coordinates
u, v = p_scaled[0] / p_scaled[2], p_scaled[1] / p_scaled[2]
print(f"Pixel coordinates: (u={u:.1f}, v={v:.1f})")

# sanity check against the closed-form equations derived above
u_formula = sample_k[0, 0] * P_c[0] / P_c[2] + sample_k[0, 2]
v_formula = sample_k[1, 1] * P_c[1] / P_c[2] + sample_k[1, 2]
print(f"Formula check:      (u={u_formula:.1f}, v={v_formula:.1f})")

K @ P_c = [2000. 1440.    5.]
Pixel coordinates: (u=400.0, v=288.0)
Formula check:      (u=400.0, v=288.0)


Both approaches agree. A point 0.5 units right and 0.3 units up, 5 units deep in camera coordinates, lands at roughly the pixel we printed above, comfortably inside a 640x480 image. Try changing `P_c` above and re-running to build intuition for how $X_c$, $Y_c$, and depth $Z_c$ each push the pixel around.

### A Simplifying Assumption: Distortion-Free Lenses

Everything above (and every `cv2` call in the rest of this primer) treats $K$ as the only thing we need to know about the inside of a camera, and every function call passes `distCoeffs=None`. That's a real simplification worth a brief discussion.

$K$ alone describes a perfect pinhole camera, where every straight line in the world stays straight in the image. Real lenses aren almost never perfect pinholes. They bend straight lines into slight curves; this is known as **radial distortion**, and it is more pronounced toward the edges of the frame, and worse on the wide field-of-view lenses common on Limelights, since a wider FOV means more bending to fit the same scene into the frame. 

It can introduce a small additional skew if the lens and sensor aren't perfectly aligned known as a **tangential distortion**.

`cv2.calibrateCamera` is the function that solves for both pieces at once from real images (typically several photos of a checkerboard pattern from different angles).Iit hands back $K$ and a small vector of distortion coefficients, the same `dist` argument this primer has been passing as `None` everywhere. A Limelight (or any calibrated vision pipeline) runs its own distortion correction internally and reports already-undistorted pixel coordinates, which is exactly what lets every $K$/$M$ equation in this notebook treat pixels as if they came from an ideal pinhole in the first place.

## Extrinsics

### Extrinsic Matrix

The intrinsic matrix above uses "camera coordinates", which we denoted with the $c$ subscript. Obviously, it is important for us to understand real-world objects from the perspective of the camera. However, we also need to understand where those objects are in the real world.

We turn to an extrinsic matrix to help us map "world coordinates" (subscript $w$) to "camera coordinates", defined as 

$$ M = [R | t] $$

where 

$$ R =
\begin{bmatrix}
r_{11} & r_{12} & r_{13} \\
r_{21} & r_{22} & r_{23} \\
r_{31} & r_{32} & r_{33} \\
\end{bmatrix}
,
t = 
\begin{bmatrix}
t_x \\
t_y \\
t_z \\
\end{bmatrix}
$$

and therefore we have 

$$ M = 

\left[
\begin{array}{ccc|c}
r_{11} & r_{12} & r_{13} & t_x \\
r_{21} & r_{22} & r_{23} & t_y \\
r_{31} & r_{32} & r_{33} & t_z \\
\end{array}
\right]
$$

The notation $[R | t]$ represents an augmented matrix, which is the matrix form of representing linear equations. So, something like $Mx$ would be written as $Rx + t$.


What do $R$ and $t$ tell us? 

#### Rotation Matrix

A **rotation matrix** rotates points in space by a specific angle without changing the size or shape of those points. A rotation can look like this: 

![](./ref_imgs/3d_rotation.png)

Naturally, we can do a number of different rotations around the different axes. 

As for what the actual values represent, they simply represent matrix multiplications that allow that rotation to take place. Differing values can create different rotations. For instance, a matrix like this one:

$$
R = 
\begin{bmatrix}
0 & 0 & -1 \\
0 & 1 & 0 \\
1 & 0 & 0 \\
\end{bmatrix}
$$

would produce the rotation we saw in the above figure. 

The top row dictates the direction of the new x-axis, the middle the y-axis, and the bottom the z-axis. 

There are a *ton* of different rotation matrices, and there is a good amount of matrix operations you would have to learn to do them properly (and understanding basis vectors!). Feel free to take time to understand the math if you so choose (`linear_algebra_primer` is a good start), but the biggest thing here is to understand underlying intuition.  


Let's apply this specific $R$ to an actual point and see where it lands:

In [4]:
R = np.array([
    [0, 0, -1],
    [0, 1,  0],
    [1, 0,  0]
])

point = np.array([1, 0, 0])  # one unit along the original x-axis
rotated_point = R @ point

print("Original point: ", point)
print("Rotated point:  ", rotated_point)

Original point:  [1 0 0]
Rotated point:   [0 0 1]


Our original x-axis point $(1, 0, 0)$ landed on $(0, 0, 1)$, putting the new point on the z-axis. That matches the picture above: this particular $R$ swaps the x- and z-axes (with a sign flip), rather than doing some arbitrary jumble of the coordinates.

#### Pitch, Yaw, and Roll

Nine numbers in a 3x3 grid is not how anyone actually *thinks* about an orientation. Pilots, roboticists, and Limelight all describe an orientation with three angles instead, each one a rotation around a single axis. We use the analogy of a plane to describe these angles. 

**Roll** represents the rotation around the x-axis. Visually, you can think of this like an airplane's wings tipping. 

**Pitch** represents the rotation around the y-axis, like the nose of the plane tilting up or down. 

**Yaw** represents rotation around the z-axis, like the plane turning left or right.

Each one of these on its own is just a rotation matrix around a single axis:

$$
R_x(\text{roll}) =
\begin{bmatrix}
1 & 0 & 0 \\
0 & \cos(\text{roll}) & -\sin(\text{roll}) \\
0 & \sin(\text{roll}) & \cos(\text{roll}) \\
\end{bmatrix}
$$
$$
\qquad
R_y(\text{pitch}) =
\begin{bmatrix}
\cos(\text{pitch}) & 0 & \sin(\text{pitch}) \\
0 & 1 & 0 \\
-\sin(\text{pitch}) & 0 & \cos(\text{pitch}) \\
\end{bmatrix}
$$
$$
\qquad
R_z(\text{yaw}) =
\begin{bmatrix}
\cos(\text{yaw}) & -\sin(\text{yaw}) & 0 \\
\sin(\text{yaw}) & \cos(\text{yaw}) & 0 \\
0 & 0 & 1 \\
\end{bmatrix}
$$

Notice each one only touches the two axes it isn't rotating around (e.g. $R_x$ leaves the x-coordinate completely alone and only mixes y and z).

A full orientation combines all three. Order matters here, since matrix multiplication doesn't commute, so rolling-then-pitching-then-yawing lands somewhere different than yawing-then-pitching-then-rolling. WPILib's `Rotation3d` (and Limelight, whose `botpose` field reports roll/pitch/yaw in the same convention) composes them as

$$
R = R_z(\text{yaw}) \, R_y(\text{pitch}) \, R_x(\text{roll})
$$

Roll is applied first, then pitch, then yaw, each one measured around the original, fixed world axes (not around whatever axis the previous rotation left behind).

In [5]:
def Rx(a):
    return np.array([
        [1, 0, 0],
        [0, np.cos(a), -np.sin(a)],
        [0, np.sin(a), np.cos(a)]
    ])

def Ry(a):
    return np.array([
        [np.cos(a), 0, np.sin(a)],
        [0, 1, 0],
        [-np.sin(a), 0, np.cos(a)]
    ])

def Rz(a):
    return np.array([
        [np.cos(a), -np.sin(a), 0],
        [np.sin(a), np.cos(a), 0],
        [0, 0, 1]
    ])

roll, pitch, yaw = np.radians([15, -10, 30])  # pick some example angles, degrees -> radians
R_from_angles = Rz(yaw) @ Ry(pitch) @ Rx(roll)

print("roll=15deg, pitch=-10deg, yaw=30deg ->")
print(R_from_angles)

roll=15deg, pitch=-10deg, yaw=30deg ->
[[ 0.85286853 -0.52188509 -0.01585001]
 [ 0.49240388  0.81404458 -0.3080095 ]
 [ 0.17364818  0.254887    0.95125124]]


Going the other direction: given some $R$ (what a PnP solve like Limelight's AprilTag pipeline actually produces), recovering the roll/pitch/yaw that built it. This is what happens internally to populate the `roll`, `pitch`, `yaw` entries of `botpose`. For our $R = R_z(\text{yaw}) \, R_y(\text{pitch}) \, R_x(\text{roll})$ order, this has a closed-form solution in terms of $R$'s entries $r_{ij}$ (using the same 1-indexed notation as the Extrinsics section above):

$$
\text{yaw} = \operatorname{atan2}(r_{21}, r_{11}) \qquad
\text{pitch} = \operatorname{atan2}(-r_{31},\ \sqrt{r_{32}^2 + r_{33}^2}) \qquad
\text{roll} = \operatorname{atan2}(r_{32}, r_{33})
$$

One edge case worth knowing about: when pitch is exactly $\pm 90°$, roll and yaw end up rotating around what has become the same physical axis, so this formula can't separate them (only their sum or difference is recoverable). This is called **gimbal lock**, and it's part of why some systems track orientation with quaternions internally even when they still report roll/pitch/yaw at the edges.

In [6]:
# decompose R_from_angles back into roll/pitch/yaw using the formulas above
r11, r21, r31 = R_from_angles[0, 0], R_from_angles[1, 0], R_from_angles[2, 0]
r32, r33 = R_from_angles[2, 1], R_from_angles[2, 2]

yaw_recovered = np.arctan2(r21, r11)
pitch_recovered = np.arctan2(-r31, np.sqrt(r32**2 + r33**2))
roll_recovered = np.arctan2(r32, r33)

print("Original (deg):  roll={:.1f}, pitch={:.1f}, yaw={:.1f}".format(*np.degrees([roll, pitch, yaw])))
print("Recovered (deg): roll={:.1f}, pitch={:.1f}, yaw={:.1f}".format(
    *np.degrees([roll_recovered, pitch_recovered, yaw_recovered])
))

Original (deg):  roll=15.0, pitch=-10.0, yaw=30.0
Recovered (deg): roll=15.0, pitch=-10.0, yaw=30.0


#### Translation Matrix

A **translation matrix** moves points in space by the same amount in a given direction, without changing the size or shape of those points. Note that while we call this a matrix, because it is 1 dimensional, it can also be referred to as a vector.

 A translation can look like this: 

![](./ref_imgs/3d_translation.png)


Just like rotations, we can do a number of different translations. 

The values are a little simpler here, since our $t$ matrix is $3 \times 1$ and our $R$ is $3 \times 3$. The top value represents the amount to move a point along the x-axis, the middle along the y-axis, and the bottom along the z-axis. 

For example, a matrix like this one 

$$
t = 
\begin{bmatrix}
0 \\
1 \\
0 \\
\end{bmatrix}
$$

will produce the above transformation. 

And applying this $t$ to a point:

In [7]:
t = np.array([0, 1, 0])

point = np.array([1, 0, 0])
translated_point = point + t

print("Original point:   ", point)
print("Translated point: ", translated_point)

Original point:    [1 0 0]
Translated point:  [1 1 0]


As expected, the point shifted by exactly $t$ along the y-axis, with the x and z coordinates untouched and the shape/size of whatever it's part of fully preserved (the defining property of a translation).

### Mapping 3D Real World to 3D Camera 

So, now that we understand the components of $M$, how does this help us go from the real-world coordinates of an object to the camera coordinates? 

$R$ tells us the rotation of our camera, and $t$ tells us the translation of the camera, both of which are relative to the real world position of the field of view that contains our object. 

We use the equation 

$$ P_c = M P_w$$

where

$$
P_c = 
\begin{bmatrix}
X_c \\
Y_c \\
Z_c \\
\end{bmatrix}, 
P_w = 
\begin{bmatrix}
X_w \\
Y_w \\
Z_w \\
1
\end{bmatrix}, 
$$

and $M$ is our same extrinsic matrix, leaving us with 

$$
\begin{bmatrix}
X_c \\
Y_c \\
Z_c \\
\end{bmatrix} = 
\left[
\begin{array}{ccc|c}
r_{11} & r_{12} & r_{13} & t_x \\
r_{21} & r_{22} & r_{23} & t_y \\
r_{31} & r_{32} & r_{33} & t_z \\
\end{array}
\right]
\begin{bmatrix}
X_w \\
Y_w \\
Z_w \\
1
\end{bmatrix}
$$





Let's combine the $R$ and $t$ from above into a full extrinsic matrix $M$ and map an actual world point through it:

In [8]:
# combine our rotation and translation into the extrinsic matrix M = [R | t]
M = np.hstack([R, t.reshape(-1, 1)])
print("Extrinsic matrix M:")
print(M)

# a point in world coordinates, in homogeneous form (with the extra 1)
P_w = np.array([8.0, 0.0, -0.5, 1.0])

P_c = M @ P_w
print("\nWorld point (X_w, Y_w, Z_w):", P_w[:3])
print("Camera point (X_c, Y_c, Z_c):", P_c)

# sanity check: M @ P_w should equal R @ P_w[:3] + t
print("R @ P_w[:3] + t:            ", R @ P_w[:3] + t)

Extrinsic matrix M:
[[ 0  0 -1  0]
 [ 0  1  0  1]
 [ 1  0  0  0]]

World point (X_w, Y_w, Z_w): [ 8.   0.  -0.5]
Camera point (X_c, Y_c, Z_c): [0.5 1.  8. ]
R @ P_w[:3] + t:             [0.5 1.  8. ]


The two calculations agree, confirming that the augmented-matrix notation $M = [R \mid t]$ really is just shorthand for $Rx + t$. Our world point $(8, 0, -0.5)$ becomes the camera-coordinate point printed above once we account for how the camera itself is rotated and positioned.

### Perspective Projection in 3D 

We have now seen that with our intrinsic matrix, we can take our coordinates from the 3D camera to the 2D image. With our extrinsic matrix, we can take our coordinates from the 3D world to the 3D camera. Unsurprisingly, we can combine the two to give us the full mapping of 3D points in the world to the 2D image.

The equation is 
$$p = \lambda KMP$$

where $\lambda$ represents the depth of our object in camera coordinates and $P$ is $P_w$ from our extrinsic equation. 

The only different term is $p$, where 

$$
p = 
\begin{bmatrix}
x_s \\
y_s \\
s
\end{bmatrix}
$$

such that $x = \frac{x_s}{s}$ and $y = \frac{y_s}{s}$, representing our 2D coordinates! 

Now you have seen how the inside and outside of the camera determine how a 3D object gets mapped to a 2D image. 


We already know $P_c$ from the extrinsic step above, so we can plug it straight into $K$ from the intrinsics section to get the full pipeline from a real-world point to a pixel:

In [9]:
p_scaled = sample_k @ P_c
u, v = p_scaled[0] / p_scaled[2], p_scaled[1] / p_scaled[2]

world_point_str = f"({P_w[0]:.1f}, {P_w[1]:.1f}, {P_w[2]:.1f})"
print(f"World point {world_point_str} -> pixel (u={u:.1f}, v={v:.1f})")

World point (8.0, 0.0, -0.5) -> pixel (u=370.0, v=340.0)


That's the complete pipeline! $P_w \to P_c$ via extrinsics, then $P_c \to$ pixel via intrinsics. 

The only reason this worked, though, is that we started out already knowing $P_w$ and walked forward through the pipeline. In practice, our Limelight only ever observes the pixel; walking backward from a pixel to a 3D point runs into the problem we're going to discuss shortly.

We also have another problem: how do we figure out this value of $\lambda$?

## Try It Yourself

Using `sample_k` from above, build a new intrinsic matrix with a larger focal length (try `f=1200`, same principal point) and reproject the same `P_c = np.array([0.5, 0.3, 5.0])` through it. How does the pixel location shift compared to our original result, and does that match the "larger f means more zoomed in" intuition from the perspective projection notebook?

Then, construct a rotation matrix that swaps the x- and y-axes (instead of x and z, like our example `R`), and apply it to `point = np.array([1, 0, 0])`. Where does it land?

In [10]:
# TODO: build a new K matrix with f=1200 (same principal point as sample_k),
# reproject P_c through it, and compare to the earlier pixel result

# TODO: build a rotation matrix that swaps the x- and y-axes,
# and apply it to point = np.array([1, 0, 0])

## Resources

- [OpenCV: Camera Calibration and 3D Reconstruction](https://docs.opencv.org/4.x/d9/d0c/group__calib3d.html) - the module that estimates both intrinsics and extrinsics from real images (`cv2.calibrateCamera`, `cv2.solvePnP`).
- [OpenCV: Camera Calibration Tutorial](https://docs.opencv.org/4.x/dc/dbb/tutorial_py_calibration.html) - a hands-on walkthrough of `cv2.calibrateCamera` against real checkerboard photos, recovering both $K$ and the distortion coefficients named above.
- [Szeliski, R. *Computer Vision: Algorithms and Applications*, Chapter 2](https://szeliski.org/Book/) - the same reference from the perspective projection notebook, with a full treatment of extrinsic parameters and rotation representations.
- [WPILib: Vision Processing](https://docs.wpilib.org/en/stable/docs/software/vision-processing/introduction/index.html) - connects these camera models to Limelight/PhotonVision setups used on FRC robots.